# RAG-01: Qwen3 embedding experiment

Small real-model check for `asshole_ai_detector.rag.embeddings` (Milestone 2, Sections 3.3 and 3.4).
Run on Colab with a GPU (T4 is enough). The setup cell clones `BRANCH`, so push the RAG-01 branch first.
Record the printed results in `docs/rag/embeddings.md`.

Local alternative: `uv sync --group rag --group data`, then run the cells after the setup cell.
Clear outputs before committing (they contain dataset text).

In [ ]:
# Colab setup. BRANCH must contain rag/embeddings.py (the RAG-01 branch until it is merged).
BRANCH = "feat/rag-01-embeddings"

import os
import sys

if not os.path.exists("/content/asshole-ai-detector"):
    !git clone -q -b {BRANCH} https://github.com/asshole-ai-detector/asshole-ai-detector.git /content/asshole-ai-detector
%cd /content/asshole-ai-detector
!pip install -q -e . "sentence-transformers>=6.3.0" "datasets>=5.2.0"

# Make the package importable without restarting the session.
sys.path.insert(0, "/content/asshole-ai-detector/src")
print(sys.version)

In [ ]:
import math
import statistics
import time

import torch

from asshole_ai_detector.data.aita_adapter import (
    iter_normalized_cases,
    load_aita_dataset,
)
from asshole_ai_detector.data.cleaning import clean_cases
from asshole_ai_detector.data.consensus import label_cleaned_case
from asshole_ai_detector.rag.embeddings import document_text, load_qwen3_embedder

start = time.perf_counter()
embedder = load_qwen3_embedder()
print(f"load: {time.perf_counter() - start:.1f}s, cuda={torch.cuda.is_available()}")
embedder.manifest()

## Sample of real cases

In [ ]:
SAMPLE_SIZE = 200

# Twice the sample size, because some posts are rejected by cleaning.
raw = load_aita_dataset(splits=("test",))
raw = raw.select(range(min(len(raw), SAMPLE_SIZE * 2)))
cases = [label_cleaned_case(c) for c in clean_cases(iter_normalized_cases(raw)).accepted]
cases = cases[:SAMPLE_SIZE]

# Token lengths of title + text, before the 1,024-token cap.
lengths = [
    len(embedder.tokenizer.encode(document_text(case), add_special_tokens=False))
    for case in cases
]
print(f"cases: {len(cases)}")
print(f"tokens: median {statistics.median(lengths)}, max {max(lengths)}")
print(f"capped (> {embedder.config.max_tokens} tokens): {sum(n > embedder.config.max_tokens for n in lengths)}")

## Document embeddings

In [ ]:
start = time.perf_counter()
vectors = embedder.embed_documents(cases)
elapsed = time.perf_counter() - start

norms = [math.sqrt(sum(v * v for v in vector)) for vector in vectors]
print(f"{len(vectors)} vectors, dims {set(map(len, vectors))}, {elapsed:.1f}s ({elapsed / len(vectors) * 1000:.0f} ms/post)")
print(f"norms: min {min(norms):.6f}, max {max(norms):.6f}")

## Query sanity check

Each query is the first two sentences of a post, rewritten only by the query prefix and lowercasing.
A working pipeline should rank the source post first for most queries. This is a smoke test, not the
retrieval relevance evaluation of Section 3.9.

In [ ]:
def first_sentences(text: str, count: int = 2) -> str:
    parts = text.replace("\n", " ").split(". ")
    return ". ".join(parts[:count])


queries = [first_sentences(case.text) for case in cases]
start = time.perf_counter()
query_vectors = embedder.embed_queries(queries)
query_ms = (time.perf_counter() - start) / len(queries) * 1000

hits = 0
for index, query in enumerate(query_vectors):
    scores = [sum(q * d for q, d in zip(query, doc)) for doc in vectors]
    hits += max(range(len(scores)), key=scores.__getitem__) == index
print(f"top-1 source match: {hits}/{len(queries)} ({hits / len(queries):.0%}), {query_ms:.0f} ms/query")